<a href="https://colab.research.google.com/github/dspraneeth07/CognitiveAttackTopology-CAT/blob/main/Notebooks/07_tmd_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================
# NOTEBOOK 07 — TOPOLOGICAL MANIPULATION DETECTION (TMD)
# CAT Framework
# CPU SAFE IMPLEMENTATION
# ===============================================================

!pip -q install pandas numpy torch networkx scikit-learn seaborn matplotlib sentence-transformers pyarrow

import torch
import json
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

from sentence_transformers import SentenceTransformer
from google.colab import drive

# ===============================================================
# MOUNT DRIVE
# ===============================================================

drive.mount('/content/drive', force_remount=True)

ROOT = Path("/content/drive/MyDrive/CAT_RESEARCH")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_DIR = ROOT/"runs"/RUN_ID
DATA_DIR = ROOT/"data"

REPORT_DIR = RUN_DIR/"reports"
PLOT_DIR = RUN_DIR/"plots"

for p in [RUN_DIR, REPORT_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Run:", RUN_ID)

# ===============================================================
# LOAD DATASET
# ===============================================================

df = pd.read_parquet(DATA_DIR/"GCT_phase1_100k.parquet")

# ===============================================================
# TEXT EMBEDDINGS (CPU SAFE)
# ===============================================================

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["text_transcript"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

embeddings = np.array(embeddings)

# ===============================================================
# TRUST SIGNAL MATRIX
# ===============================================================

trust_signals = df[
["urgency_score","fear_trigger_score","authority_claim","trust_manipulation_score"]
].values

# ===============================================================
# TRUST GRAPH CONSTRUCTION
# ===============================================================

G = nx.Graph()

senders = df["ip_address_hash"]
users = df["interaction_id"]

edges = list(zip(senders,users))

G.add_edges_from(edges)

# ===============================================================
# GRAPH FEATURES
# ===============================================================

degree = nx.degree_centrality(G)

deg_feature = np.array([degree.get(u,0) for u in users])

clustering = nx.clustering(G)

cluster_feature = np.array([clustering.get(u,0) for u in users])

# ===============================================================
# TEMPORAL TRUST VARIANCE (KALMAN)
# ===============================================================

class KalmanVariance:

    def __init__(self):

        self.x = 1
        self.P = 1
        self.Q = 1e-5
        self.R = 1e-2

    def update(self,z):

        K = self.P/(self.P+self.R)

        self.x = self.x + K*(z-self.x)

        self.P = (1-K)*self.P + self.Q

        return self.x

kf = KalmanVariance()

sigma_trust = trust_signals.std(axis=1)

sigma_smooth = np.array([kf.update(v) for v in sigma_trust])

# ===============================================================
# LOAD TRUST TENSOR
# ===============================================================

tensor_path = list(ROOT.glob("runs/*/tensor/trust_tensor.pt"))[-1]

T = torch.load(tensor_path)

users_n,channels,signals = T.shape

# ===============================================================
# CDE (tensor-first)
# ===============================================================

weights = torch.ones_like(T)/signals

CDE = torch.einsum("ijk,ijk->i",T,weights).detach().numpy()

# ===============================================================
# TRUST DISTORTION INDEX
# ===============================================================

# Fix for ValueError: operands could not be broadcast together with shapes
# CDE has length users_n (35574), sigma_smooth has length len(df) (100000)
# To make them compatible for division and later hstack, we will pad TDI
# after computing it for the available length.

# Calculate TDI for the shorter length (users_n)
TDI_short = CDE / sigma_smooth[:users_n]

# Create a full-length TDI array, padded with zeros (or a suitable default)
TDI = np.zeros(len(df))
TDI[:users_n] = TDI_short

# ===============================================================
# FEATURE FUSION
# ===============================================================

X = np.hstack([
embeddings,
trust_signals,
deg_feature.reshape(-1,1),
cluster_feature.reshape(-1,1),
TDI.reshape(-1,1)
])

y = df["human_verified_label"].values

# ===============================================================
# TRAIN TEST SPLIT
# ===============================================================

X_train,X_test,y_train,y_test = train_test_split(
X,y,test_size=0.2,random_state=42,stratify=y
)

# ===============================================================
# CLASSIFIER
# ===============================================================

model = RandomForestClassifier(n_estimators=200,n_jobs=-1)

model.fit(X_train,y_train)

pred = model.predict(X_test)

prob = model.predict_proba(X_test)[:,1]

# ===============================================================
# METRICS
# ===============================================================

roc = roc_auc_score(y_test,prob)

prec = precision_score(y_test,pred)

rec = recall_score(y_test,pred)

f1 = f1_score(y_test,pred)

metrics = pd.DataFrame({

"ROC_AUC":[roc],
"Precision":[prec],
"Recall":[rec],
"F1":[f1]

})

metrics.to_csv(REPORT_DIR/"tmd_model_results.csv",index=False)

# ===============================================================
# ROC CURVE
# ===============================================================

from sklearn.metrics import roc_curve

fpr,tpr,_ = roc_curve(y_test,prob)

plt.figure()

plt.plot(fpr,tpr)

plt.title("TMD ROC Curve")

plt.savefig(PLOT_DIR/"tmd_roc_curve.png")

# ===============================================================
# PR CURVE
# ===============================================================

from sklearn.metrics import precision_recall_curve

precision,recall,_ = precision_recall_curve(y_test,prob)

plt.figure()

plt.plot(recall,precision)

plt.title("TMD Precision Recall")

plt.savefig(PLOT_DIR/"tmd_pr_curve.png")

# ===============================================================
# FINAL REPORT
# ===============================================================

report = {

"roc_auc":float(roc),
"precision":float(prec),
"recall":float(rec),
"f1":float(f1)

}

with open(REPORT_DIR/"tmd_detection_report.json","w") as f:

    json.dump(report,f,indent=4)

print("Notebook 07 completed")